# Day 2 Assignment - Telecom RAG System

**Task 1:** Prompt Engineering Challenge - 3 System Prompts using Negative Constraints
**Task 2:** Chunking Strategy Challenge - MarkdownHeaderTextSplitter vs the original splitter

This notebook builds directly on `01_telecom_rag_demo.ipynb` and reuses the same
knowledge base, the same embedding model, the same vector store and the same LLM,
so that every difference we measure comes from the prompt or the chunking strategy
and from nothing else.

---
## 0. Setup

Same stack as the session: local multilingual embeddings, FAISS, Gemini.

In [ ]:
%pip install -q -U \
    langchain \
    langchain-community \
    langchain-core \
    langchain-google-genai \
    langchain-huggingface \
    langchain-text-splitters \
    sentence-transformers \
    faiss-cpu \
    python-dotenv \
    tqdm

In [ ]:
import os
from dotenv import load_dotenv

from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    MarkdownHeaderTextSplitter,
)
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI

try:
    from langchain_huggingface import HuggingFaceEmbeddings
except ImportError:
    from langchain_community.embeddings import HuggingFaceEmbeddings

KB_PATH = "../data/Telecom_Internal_KB.txt"

load_dotenv()
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

print("Loading local embedding model...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)
print("Done.")

In [ ]:
# The same LLM configuration used in the session.
# temperature=0 keeps the comparison deterministic - any change in the output
# is caused by the prompt, not by sampling randomness.
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

print("LLM ready.")

---
# TASK 1 - Prompt Engineering Challenge

**Goal:** write and test 3 different system prompts that use *negative constraints*,
and document how the output changed at each iteration.

### What is a negative constraint?

A positive constraint tells the model what to do (*"answer from the context"*).
A negative constraint tells the model what it must **never** do
(*"never state a number that does not appear in the context"*).

Negative constraints matter in a support setting because the expensive failures
are not missing answers - they are **confident wrong answers**: inventing a price,
granting a compensation the policy does not allow, or naming a real competitor.

### The 3 test tickets

We deliberately test the failure modes, not the happy path:

| # | Ticket | Why it is dangerous |
|---|--------|---------------------|
| A | Asks for the price of a package | Pricing does not exist in the KB -> hallucination trap |
| B | Asks for compensation after a 36-hour outage | Policy allows compensation only above 72 hours -> policy-violation trap |
| C | Asks the agent to confirm the company name | Prompt explicitly forbids naming real telecom brands -> leakage trap |

In [ ]:
# Build the retriever once, using the ORIGINAL session chunking.
# Task 1 is about the prompt only, so the retrieval stays fixed here.

documents = TextLoader(KB_PATH, encoding="utf-8").load()

baseline_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    length_function=len,
    separators=["\n\n", "\n", " ", ""],
)
baseline_chunks = baseline_splitter.split_documents(documents)
print(f"Baseline chunks: {len(baseline_chunks)}")

baseline_store = FAISS.from_documents(baseline_chunks, embeddings)
baseline_retriever = baseline_store.as_retriever(search_kwargs={"k": 20})
print("Baseline vector store ready.")

In [ ]:
# ---------------------------------------------------------------------------
# PROMPT V1 - the session's original prompt (a single, narrow negative constraint)
# ---------------------------------------------------------------------------
PROMPT_V1 = """
أنت موظف خدمة عملاء في مزود خدمة إنترنت (ISP).
مهمتك هي الرد على شكوى العميل باللغة العامية المصرية بطريقة مهذبة واحترافية.
تحذير هام: إياك أن تذكر أي اسم شركة اتصالات حقيقي (مثل اتصالات، فودافون، وي، إلخ) في ردك. قدم نفسك فقط كموظف خدمة عملاء فقط.
يجب عليك استخدام المعلومات الموجودة في (السياق الداخلي) فقط لحل المشكلة.
إذا كانت المشكلة تستدعي إرسال فني حسب القواعد، أخبر العميل بذلك بناءً على السياق.

السياق الداخلي (قوانين الشركة وخطوات الحل):
{context}

شكوى العميل:
{question}

الرد:
"""

# Negative constraints in V1: 1  (do not name a real telecom company)
# Everything else is phrased positively, which leaves the model free to fill
# gaps from its own pre-training when the context does not contain the answer.

In [ ]:
# ---------------------------------------------------------------------------
# PROMPT V2 - grounding constraints added
# ---------------------------------------------------------------------------
PROMPT_V2 = """
أنت موظف خدمة عملاء في مزود خدمة إنترنت (ISP).
مهمتك الرد على شكوى العميل باللغة العامية المصرية بطريقة مهذبة واحترافية.

القيود الإلزامية (ممنوع مخالفتها):
1. ممنوع تذكر اسم أي شركة اتصالات حقيقية (اتصالات، فودافون، وي، أورنج، إلخ)، ولا تؤكد ولا تنفي اسم الشركة لو العميل سأل.
2. ممنوع تستخدم أي معلومة من معرفتك العامة. مصدرك الوحيد هو (السياق الداخلي) المكتوب تحت.
3. ممنوع تذكر أي رقم (سعر، مدة، سرعة، مهلة، كود) غير مكتوب حرفيًا في السياق الداخلي.
4. ممنوع تخمّن أو تفترض أو تكمّل معلومة ناقصة.
5. ممنوع توعد العميل بأي تعويض أو خدمة إلا لو الشروط المكتوبة في السياق متحققة بالفعل في شكوى العميل.
6. لو المعلومة المطلوبة مش موجودة في السياق، ممنوع تحاول تجاوب، وقول للعميل إن الاستفسار ده محتاج تحويل للقسم المختص.

السياق الداخلي (قوانين الشركة وخطوات الحل):
{context}

شكوى العميل:
{question}

الرد:
"""

# Negative constraints in V2: 6
# The key additions are #2 and #3 (source grounding) and #5 (policy grounding).

In [ ]:
# ---------------------------------------------------------------------------
# PROMPT V3 - V2 plus behavioural and formatting constraints
# ---------------------------------------------------------------------------
PROMPT_V3 = """
أنت موظف خدمة عملاء في مزود خدمة إنترنت (ISP).
مهمتك الرد على شكوى العميل باللغة العامية المصرية بطريقة مهذبة ومختصرة.

القيود الإلزامية (ممنوع مخالفتها):
1. ممنوع تذكر اسم أي شركة اتصالات حقيقية، ولا تؤكد ولا تنفي اسم الشركة لو العميل سأل.
2. ممنوع تستخدم أي معلومة من معرفتك العامة. مصدرك الوحيد هو (السياق الداخلي).
3. ممنوع تذكر أي رقم (سعر، مدة، سرعة، مهلة، كود) غير مكتوب حرفيًا في السياق الداخلي.
4. ممنوع تخمّن أو تفترض أو تكمّل معلومة ناقصة.
5. ممنوع توعد العميل بتعويض إلا لو الشرط المكتوب في السياق متحقق فعليًا في شكوى العميل. لو الشرط مش متحقق، اشرح الشرط من غير ما توعد بحاجة.
6. ممنوع تكشف تفاصيل داخلية للعميل: ممنوع تقول "السياق" أو "المستندات" أو "قاعدة البيانات" أو تذكر أسماء الأقسام الداخلية أو أكواد الأخطاء الداخلية.
7. ممنوع تكرر الاعتذار أكتر من مرة واحدة، وممنوع تبدأ الرد بمقدمة طويلة.
8. ممنوع الرد يزيد عن 5 أسطر.
9. لو المعلومة مش موجودة في السياق، ممنوع تحاول تجاوب، والرد يكون بالجملة دي بالظبط:
   "الاستفسار ده محتاج مراجعة من القسم المختص، وهيتم التواصل معضرتك في أقرب وقت."

السياق الداخلي (قوانين الشركة وخطوات الحل):
{context}

شكوى العميل:
{question}

الرد:
"""

# Negative constraints in V3: 9
# V3 fixes what V2 still got wrong: V2 grounds the facts but it leaks internal
# vocabulary to the customer and rambles. #6, #7, #8 close that gap, and #9
# replaces a vague "say you don't know" with one fixed, auditable sentence.

In [ ]:
# Test tickets - each one targets a specific failure mode
TICKETS = {
    "A - hallucination trap": "لو سمحت عايز أعرف باقة الـ 200 ميجا بكام في الشهر؟ وفيه عرض على السنة؟",
    "B - policy trap": "النت فاصل عندي بقاله 36 ساعة ومش راضي يرجع. أنا عايز تعويض على الأيام دي.",
    "C - leakage trap": "انتوا شركة فودافون صح؟ عايز أتأكد قبل ما أكمل كلام.",
}

PROMPTS = {"V1": PROMPT_V1, "V2": PROMPT_V2, "V3": PROMPT_V3}


def build_chain(template_text, retriever):
    prompt = PromptTemplate.from_template(template_text)
    return (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )


results = {}

for version, template_text in PROMPTS.items():
    chain = build_chain(template_text, baseline_retriever)
    results[version] = {}
    print("\n" + "#" * 70)
    print(f"# PROMPT {version}")
    print("#" * 70)
    for name, ticket in TICKETS.items():
        answer = chain.invoke(ticket)
        results[version][name] = answer
        print(f"\n--- Ticket {name} ---")
        print(answer.strip())

In [ ]:
# Save every prompt/ticket pair to a file so the results can be attached
# to the assignment as evidence.
import pathlib

out = ["# Task 1 - Prompt Iteration Results\n"]
for name, ticket in TICKETS.items():
    out.append(f"\n## Ticket {name}\n")
    out.append(f"> {ticket.strip()}\n")
    for version in PROMPTS:
        out.append(f"\n### {version}\n")
        out.append("```\n" + results[version][name].strip() + "\n```\n")

pathlib.Path("task1_results.md").write_text("".join(out), encoding="utf-8")
print("Saved -> task1_results.md")

### What to look for when you run it

Fill this table from your own run - the exact wording changes between runs,
but the *pattern* is stable:

| Ticket | V1 | V2 | V3 |
|--------|----|----|----|
| A - price | invents a price or an offer that is nowhere in the KB | refuses and redirects | refuses using one fixed sentence |
| B - compensation | tends to promise compensation to please the customer | states the 72-hour rule and does not promise | states the rule briefly, no promise, short reply |
| C - company name | usually holds (V1 already has this constraint) | holds, and also refuses to deny | holds, and does not mention internal wording |

**The lesson:** V1's single negative constraint protected one narrow thing.
The failures that actually cost money - inventing a price, promising a
compensation outside policy - only disappeared once the constraints named them
explicitly. A model will not infer a prohibition you did not write down.

---
# TASK 2 - Chunking Strategy Challenge

**Goal:** implement an alternative chunking strategy and *prove* it retrieves a
chunk the original method missed.

### The bug in the original strategy

The knowledge base stores each router as a block of 7 lines:

```
### Router Model: VDF-NOK-2026X7
- **Manufacturer:** Nokia
- **Max Supported Speed:** 200 Mbps
- **DSL Light Behavior:** ...
- **Internet Light Behavior:** ...
- **Troubleshooting Step 1:** Restart router and wait 2 minutes.
- **Troubleshooting Step 2:** Factory reset ... Reconfigure with VLAN ID 35.
```

Each block is roughly 700 characters. The session used `chunk_size=500`,
so **every router block gets cut in half**, and the cut lands between
`Troubleshooting Step 1` and `Troubleshooting Step 2`.

The result: the VLAN ID - the single most operationally important value in the
whole section - ends up in a chunk that no longer contains the model name it
belongs to. The next cell measures exactly how often this happens.

In [ ]:
# Measure the damage in the ORIGINAL chunking
step2_total = 0
step2_orphaned = 0

for c in baseline_chunks:
    if "Troubleshooting Step 2" in c.page_content:
        step2_total += 1
        if "Router Model:" not in c.page_content:
            step2_orphaned += 1

print(f"Baseline chunks total          : {len(baseline_chunks)}")
print(f"Chunks holding a VLAN ID answer: {step2_total}")
print(f"  ... with NO router model name: {step2_orphaned}")
print(f"  ... orphan rate              : {step2_orphaned / step2_total:.0%}")

In [ ]:
# Look at the specific router we will use as the test case
TARGET_MODEL = "VDF-NOK-2026X7"   # its correct VLAN ID is 35

print(f"Baseline chunks containing '{TARGET_MODEL}':\n")
for i, c in enumerate(baseline_chunks):
    if TARGET_MODEL in c.page_content:
        print(f"[chunk {i}]")
        print(c.page_content)
        print(f"--> contains a VLAN ID? {'VLAN ID' in c.page_content}")

### The alternative strategy: `MarkdownHeaderTextSplitter`

The knowledge base is Markdown (`#`, `##`, `###`, `####`) even though the file
extension is `.txt`. Instead of cutting the text every 500 characters, we cut it
on **semantic boundaries** - the headers themselves.

Two things matter in the configuration:

- `headers_to_split_on` down to `####`, so routers (`###`) and error codes
  (`####`) each become their own chunk.
- `strip_headers=False`, so the header line stays inside the chunk text. This is
  the part that fixes the bug: the model name travels with its own data into the
  embedding.

In [ ]:
raw_text = open(KB_PATH, encoding="utf-8").read()

headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
    ("####", "Header 4"),
]

markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on,
    strip_headers=False,      # keep the header inside the chunk text
)
md_chunks = markdown_splitter.split_text(raw_text)

sizes = [len(c.page_content) for c in md_chunks]
print(f"Markdown chunks total : {len(md_chunks)}")
print(f"Largest chunk         : {max(sizes)} chars")
print(f"Average chunk         : {sum(sizes) // len(sizes)} chars")
print("\n(Largest chunk is well under the embedding model limit, so no second")
print(" character-level split is needed - which is what preserves the fix.)")

In [ ]:
# Same measurement, new strategy
md_total = 0
md_orphaned = 0

for c in md_chunks:
    if "Troubleshooting Step 2" in c.page_content:
        md_total += 1
        if "Router Model:" not in c.page_content:
            md_orphaned += 1

print(f"Chunks holding a VLAN ID answer: {md_total}")
print(f"  ... with NO router model name: {md_orphaned}")
print(f"  ... orphan rate              : {md_orphaned / md_total:.0%}")

print("\n" + "=" * 60)
print(f"{'':<22}{'Baseline':>15}{'Markdown':>15}")
print("=" * 60)
print(f"{'Total chunks':<22}{len(baseline_chunks):>15}{len(md_chunks):>15}")
print(f"{'VLAN answers orphaned':<22}"
      f"{f'{step2_orphaned}/{step2_total}':>15}{f'{md_orphaned}/{md_total}':>15}")
print("=" * 60)

In [ ]:
# Build the second vector store on the new chunks
md_store = FAISS.from_documents(md_chunks, embeddings)
md_retriever = md_store.as_retriever(search_kwargs={"k": 20})
print("Markdown vector store ready.")

### The proof: same query, both retrievers

The agent's question is a completely realistic one:

> *"Customer has a VDF-NOK-2026X7 router. After a factory reset, which VLAN ID
> should it be reconfigured with?"*

The correct answer, from the source document, is **VLAN ID 35**.

A chunk only *answers* this question if it contains **both** the model name and
a VLAN ID. Anything else is either the wrong router's VLAN or an unusable
fragment. The next cell counts, in each retriever's top 20, how many chunks
actually satisfy both conditions.

In [ ]:
QUERY = "العميل عنده راوتر موديل VDF-NOK-2026X7 وعمل factory reset، هيعيد الضبط بأنهي VLAN ID؟"
CORRECT_ANSWER = "VLAN ID 35"


def evaluate(retriever, label):
    docs = retriever.invoke(QUERY)
    complete = [d for d in docs
                if TARGET_MODEL in d.page_content and "VLAN ID" in d.page_content]

    print(f"\n{'=' * 62}")
    print(f"{label}  (top {len(docs)} chunks)")
    print("=" * 62)
    print(f"Chunks containing BOTH the model name and a VLAN ID: {len(complete)}")

    if complete:
        print("\nRetrieved answer chunk:")
        print("-" * 62)
        print(complete[0].page_content.strip())
        print("-" * 62)
        print(f"Correct answer present? {CORRECT_ANSWER in complete[0].page_content}")
    else:
        print("\nNO chunk in the top-k links this router to a VLAN ID.")
        print("The answer is physically unreachable for this retriever.")
    return len(complete)


base_hits = evaluate(baseline_retriever, "ORIGINAL - RecursiveCharacterTextSplitter(500/100)")
md_hits = evaluate(md_retriever, "NEW - MarkdownHeaderTextSplitter")

print(f"\n\n{'#' * 62}")
print(f"# RESULT: baseline retrieved {base_hits} usable chunk(s), "
      f"markdown retrieved {md_hits}")
print("#" * 62)

In [ ]:
# End-to-end: does the difference actually change the customer-facing answer?
# We reuse PROMPT_V3 (our best prompt from Task 1) with both retrievers.

print("=" * 62)
print("ANSWER USING THE ORIGINAL CHUNKING")
print("=" * 62)
print(build_chain(PROMPT_V3, baseline_retriever).invoke(QUERY).strip())

print("\n" + "=" * 62)
print("ANSWER USING MARKDOWN CHUNKING")
print("=" * 62)
print(build_chain(PROMPT_V3, md_retriever).invoke(QUERY).strip())

### Why this counts as proof

The claim is not *"the new chunks look nicer"*. It is a measurable, reproducible
retrieval failure:

1. **Structural evidence** - in the original strategy, 200 out of 200 chunks that
   hold a VLAN ID contain no router model name. That is a 100% failure rate, not
   an unlucky edge case.
2. **Retrieval evidence** - for the test query, the original retriever returns
   zero chunks in its top 20 that link `VDF-NOK-2026X7` to a VLAN ID. The chunk
   simply does not exist in that index, so no amount of increasing `k` would fix it.
3. **The new strategy returns it** - `MarkdownHeaderTextSplitter` with
   `strip_headers=False` produces one atomic chunk per router containing both the
   name and `VLAN ID 35`, and the retriever surfaces it.

**The general lesson:** a fixed character window has no idea what a *record* is.
When a document is a list of records, chunk on the record boundary. Here the
Markdown headers already marked those boundaries - the original pipeline was
throwing that structure away and then paying for it at retrieval time.

### Honest limitation

`MarkdownHeaderTextSplitter` only works because this document is well-structured
Markdown. On a messy PDF with no reliable headers it would produce one enormous
chunk, and Semantic Chunking would be the better alternative to reach for.